# LeetCode 3920 — Maximize Fixed Points After Deletions bằng Python

- Link: https://leetcode.com/problems/maximize-fixed-points-after-deletions/
- Độ khó: **Hard**
- Chủ đề: Mảng, quy hoạch động, dãy con tăng dài nhất (LIS) hai chiều, tìm kiếm nhị phân, Fenwick tree

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxFixedPoints(self, nums: list[int]) -> int:
```

## 1. Mục tiêu học

1. Biết cách "dịch" một thao tác khó hình dung (xóa phần tử làm chỉ số dồn trái) thành **điều kiện trên các phần tử được giữ lại**.
2. Phát hiện đại lượng bất biến `d = i - nums[i]` (số phần tử *bị xóa* trước vị trí đó) và vì sao nó phải **không giảm**.
3. Đưa bài về **LIS hai chiều**: chọn dãy dài nhất có `nums` tăng ngặt và `d` không giảm.
4. Cài đặt LIS `O(n log n)` bằng `bisect` (patience sorting) và bằng **Fenwick tree lấy max**.
5. Viết test đối chiếu với vét cạn mọi tập xóa.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums`. Vị trí `i` là **điểm bất động** nếu `nums[i] == i`. Ta được xóa bao nhiêu phần tử tùy ý (kể cả 0). Sau khi xóa, các phần tử còn lại dồn sang trái và được đánh chỉ số lại từ 0. Hỏi số điểm bất động **lớn nhất** có thể đạt được.

**Ví dụ 1:** `nums = [0,2,1]` → `2`. Xóa `2` được `[0,1]`, cả hai vị trí bất động.

**Ví dụ 2:** `nums = [3,1,2]` → `2`. Không xóa gì: `nums[1] = 1`, `nums[2] = 2`.

**Ví dụ 3:** `nums = [1,0,1,2]` → `3`. Xóa phần tử đầu được `[0,1,2]`.

**Ràng buộc:** `1 ≤ nums.length ≤ 10^5`, `0 ≤ nums[i] ≤ 10^5`.

In [1]:
from typing import List
from bisect import bisect_left
from itertools import combinations
import random, time

examples = [([0, 2, 1], 2), ([3, 1, 2], 2), ([1, 0, 1, 2], 3)]
def count_fixed(arr):
    return sum(1 for i, x in enumerate(arr) if x == i)

for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp, "| số điểm bất động nếu không xóa:", count_fixed(nums))

[0, 2, 1] -> kỳ vọng 2 | số điểm bất động nếu không xóa: 1
[3, 1, 2] -> kỳ vọng 2 | số điểm bất động nếu không xóa: 2
[1, 0, 1, 2] -> kỳ vọng 3 | số điểm bất động nếu không xóa: 0


## 3. Kiến thức nền

### 3.1. `enumerate` và list comprehension

`enumerate(nums)` sinh các cặp `(chỉ_số, giá_trị)`. Kết hợp list comprehension để tạo nhanh các mảng phụ như `d[i] = i - nums[i]`.

In [2]:
nums = [1, 0, 1, 2]
d = [i - x for i, x in enumerate(nums)]
print("d =", d)
print("các vị trí có thể thành bất động (d >= 0):", [i for i in range(len(nums)) if d[i] >= 0])

d = [-1, 1, 1, 1]
các vị trí có thể thành bất động (d >= 0): [1, 2, 3]


### 3.2. Sắp xếp theo nhiều khóa

`sorted(items)` với `items` là list tuple sẽ sắp theo phần tử đầu, rồi phần tử thứ hai khi bằng nhau… Muốn khóa giảm dần thì đổi dấu: `key=lambda p: (p[0], -p[1])`.

In [3]:
items = [(1, 5), (0, 3), (1, 2), (0, 7)]
print(sorted(items))
print(sorted(items, key=lambda p: (p[0], -p[1])))

[(0, 3), (0, 7), (1, 2), (1, 5)]
[(0, 7), (0, 3), (1, 5), (1, 2)]


### 3.3. LIS `O(n log n)` với `bisect_left`

Giữ mảng `tails`, trong đó `tails[L]` = giá trị cuối **nhỏ nhất** của một dãy con tăng ngặt độ dài `L+1`. Với mỗi `x`: tìm `pos = bisect_left(tails, x)`; nếu `pos == len(tails)` thì nối thêm, ngược lại thay `tails[pos] = x`. Độ dài LIS = `len(tails)`.

- `bisect_left` → dãy **tăng ngặt**.
- `bisect_right` → dãy **không giảm**.

In [4]:
def lis_strict(seq):
    tails = []
    for x in seq:
        pos = bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
    return len(tails)

print(lis_strict([3, 1, 2, 2, 5, 4, 6]))   # 1,2,4,6 hoặc 1,2,5,6 -> 4

4


### 3.4. Fenwick tree (Binary Indexed Tree) lấy max tiền tố

Fenwick tree hỗ trợ: cập nhật `tree[i] = max(tree[i], val)` và truy vấn `max(tree[1..i])`, mỗi thao tác `O(log N)`. Chỉ dùng được khi giá trị tại một vị trí **chỉ tăng** (đúng với DP LIS).

In [5]:
class FenwickMax:
    def __init__(self, size):
        self.n = size
        self.t = [0] * (size + 1)
    def update(self, i, val):      # i: 1-indexed
        while i <= self.n:
            if self.t[i] < val:
                self.t[i] = val
            i += i & -i
    def query(self, i):            # max trên [1..i]
        res = 0
        while i > 0:
            if self.t[i] > res:
                res = self.t[i]
            i -= i & -i
        return res

fw = FenwickMax(10)
fw.update(3, 5); fw.update(7, 2)
print(fw.query(2), fw.query(3), fw.query(10))

0 5 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thay vì nghĩ "xóa", hãy nghĩ "**giữ lại** một dãy con". Phần tử ở chỉ số gốc `i` có giá trị `x` sẽ thành điểm bất động nếu **đúng `x` phần tử được giữ ở trước nó**.

- Trước nó chỉ có `i` phần tử gốc, nên cần `x ≤ i`. Đặt `d = i - x`: đây chính là **số phần tử bị xóa** trước nó. Cần `d ≥ 0`.
- Nếu hai phần tử `i < j` cùng là điểm bất động, chỉ số mới của chúng là `x_i` và `x_j` nên phải có `x_i < x_j` (**giá trị tăng ngặt**).
- Số phần tử bị xóa trước `j` ≥ số bị xóa trước `i` (xóa thì chỉ cộng dồn) ⇒ `d_i ≤ d_j` (**d không giảm**).

Tính tay ví dụ 3, `nums = [1,0,1,2]`:

| i | x = nums[i] | d = i − x | dùng được? |
|---|---|---|---|
| 0 | 1 | −1 | không |
| 1 | 0 | 1 | có |
| 2 | 1 | 1 | có |
| 3 | 2 | 1 | có |

Chọn cả 3 phần tử cuối: `x = 0,1,2` tăng ngặt, `d = 1,1,1` không giảm ⇒ đáp án 3 (xóa đúng 1 phần tử ở đầu).

In [6]:
for nums, exp in examples:
    rows = [(i, x, i - x) for i, x in enumerate(nums)]
    print(nums, "->", rows)

[0, 2, 1] -> [(0, 0, 0), (1, 2, -1), (2, 1, 1)]
[3, 1, 2] -> [(0, 3, -3), (1, 1, 0), (2, 2, 0)]
[1, 0, 1, 2] -> [(0, 1, -1), (1, 0, 1), (2, 1, 1), (3, 2, 1)]


## 5. Cách 1 — Brute Force: thử mọi tập phần tử giữ lại

**Ý tưởng.** Với mỗi tập con chỉ số được giữ (theo thứ tự gốc), dựng mảng mới và đếm điểm bất động. Có `2^n` tập con, mỗi tập tốn `O(n)`.

In [7]:
def brute(nums):
    n = len(nums)
    best = 0
    for mask in range(1 << n):
        kept = [nums[i] for i in range(n) if mask >> i & 1]
        best = max(best, count_fixed(kept))
    return best

for nums, exp in examples:
    print(brute(nums), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** `O(2^n · n)`. Với `n = 10^5` là không tưởng.

**Nút thắt:** ta đang xét cả những phần tử "không quan trọng" (những phần tử không thành điểm bất động). Thực ra chỉ cần quyết định **tập điểm bất động**; các phần tử khác giữ hay xóa có thể tự điều chỉnh để chỉ số khớp.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề (điều kiện cần và đủ).** Một tập chỉ số `i_1 < i_2 < … < i_t` có thể **đồng thời** là điểm bất động ⇔

1. `d_{i_j} = i_j - nums[i_j] ≥ 0` với mọi `j`;
2. `nums[i_1] < nums[i_2] < … < nums[i_t]` (tăng ngặt);
3. `d_{i_1} ≤ d_{i_2} ≤ … ≤ d_{i_t}` (không giảm).

*Đủ:* giữa `i_j` và `i_{j+1}` có `i_{j+1} - i_j - 1` phần tử gốc, ta cần giữ đúng `x_{j+1} - x_j - 1` phần tử trong đó. Điều kiện 3 ⇔ `x_{j+1} - x_j ≤ i_{j+1} - i_j` nên đủ phần tử để giữ; điều kiện 2 bảo đảm số cần giữ ≥ 0. Trước `i_1` giữ đúng `x_1 ≤ i_1` phần tử (điều kiện 1).

**Bước đột phá:** chú ý `i = d + x`. Nếu `x` tăng ngặt và `d` không giảm thì `i` **tự động tăng ngặt**. Vậy ta **quên chỉ số `i`** và chỉ cần:

> Chọn nhiều điểm nhất trong các cặp `(d, x)` (với `d ≥ 0`) sao cho `d` không giảm và `x` tăng ngặt.

Đây là **LIS hai chiều**: sắp các cặp theo `(d tăng, x tăng)` rồi tìm dãy con **tăng ngặt** theo `x`. (Hai cặp cùng `d` và cùng `x` không thể xảy ra vì khi đó cùng `i`.)

## 7. Các cách tiếp theo

- **Cách 2:** DP `O(n²)` trực tiếp trên chỉ số: `dp[i] = 1 + max dp[j]` với `j < i`, `nums[j] < nums[i]`, `d_j ≤ d_i`.
- **Cách 3:** sắp xếp + LIS bằng `bisect` — `O(n log n)` (lời giải nộp chính).
- **Cách 4:** duyệt theo `d` tăng dần + Fenwick tree max trên trục `x` — `O(n log V)`.

### 7.1. Cách 2 — DP `O(n²)`

**Giải thích:** `dp[i]` = số điểm bất động nhiều nhất trong một tập hợp lệ **kết thúc** ở `i` (tức `i` là điểm bất động cuối cùng). Chỉ xét `i` có `d_i ≥ 0`.

In [8]:
def dp_quadratic(nums):
    n = len(nums)
    d = [i - x for i, x in enumerate(nums)]
    dp = [0] * n
    best = 0
    for i in range(n):
        if d[i] < 0:
            continue
        dp[i] = 1
        for j in range(i):
            if d[j] >= 0 and nums[j] < nums[i] and d[j] <= d[i]:
                dp[i] = max(dp[i], dp[j] + 1)
        best = max(best, dp[i])
    return best

for nums, exp in examples:
    print(dp_quadratic(nums), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


In [9]:
class SolutionQuadratic:
    def maxFixedPoints(self, nums: list[int]) -> int:
        n = len(nums)
        dp = [0] * n
        best = 0
        for i in range(n):
            di = i - nums[i]
            if di < 0:
                continue
            cur = 1
            for j in range(i):
                dj = j - nums[j]
                if 0 <= dj <= di and nums[j] < nums[i] and dp[j] + 1 > cur:
                    cur = dp[j] + 1
            dp[i] = cur
            best = max(best, cur)
        return best

print([SolutionQuadratic().maxFixedPoints(x) for x, _ in examples])

[2, 2, 3]


**Độ phức tạp Cách 2:** `O(n²)` thời gian, `O(n)` bộ nhớ — quá chậm cho `n = 10^5` (`10^{10}` phép so sánh).

### 7.2. Cách 3 — Sắp xếp + LIS bằng `bisect` (lời giải chính)

**Giải thích từng đoạn:**

1. Tạo danh sách cặp `(d, x)` với `d = i - x ≥ 0`.
2. Sắp xếp **tăng theo `d`, khi bằng nhau thì tăng theo `x`**. Khi `d` bằng nhau, các `x` khác nhau tăng dần đều có thể chọn hết (cùng `d`, `x` tăng ⇒ hợp lệ) — sắp `x` tăng giúp LIS lấy được tất cả.
3. Chạy LIS tăng ngặt trên dãy `x` sau khi sắp.

Vì sao không bị "chọn sai" hai cặp `d` lớn trước `d` nhỏ? Vì sau khi sắp, thứ tự trong dãy chính là thứ tự `d` không giảm, nên mọi dãy con đều thỏa điều kiện `d`.

In [10]:
def lis_solution(nums):
    pairs = sorted((i - x, x) for i, x in enumerate(nums) if i - x >= 0)
    tails = []
    for _, x in pairs:
        pos = bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
    return len(tails)

for nums, exp in examples:
    print(lis_solution(nums), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp Cách 3:** sắp xếp `O(n log n)` + LIS `O(n log n)` ⇒ `O(n log n)`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — Duyệt theo `d` tăng dần + Fenwick tree max

**Giải thích:** cũng xử lý các cặp theo `(d, x)` tăng dần. Với cặp `(d, x)`: `best = 1 + max{dp của các cặp đã xử lý có giá trị < x}` = `1 + query(x)` (dùng chỉ số Fenwick `x + 1`, truy vấn tiền tố `[1..x]` tương ứng giá trị `0..x-1`). Sau đó `update(x + 1, best)`.

Vì các cặp cùng `d` được xử lý theo `x` tăng, cặp sau có thể nối tiếp cặp trước cùng `d` — đúng như điều kiện "d không giảm".

In [11]:
def fenwick_solution(nums):
    pairs = sorted((i - x, x) for i, x in enumerate(nums) if i - x >= 0)
    if not pairs:
        return 0
    V = max(x for _, x in pairs) + 2
    fw = FenwickMax(V)
    ans = 0
    for _, x in pairs:
        best = fw.query(x) + 1         # các giá trị 0..x-1  <->  chỉ số 1..x
        fw.update(x + 1, best)
        ans = max(ans, best)
    return ans

for nums, exp in examples:
    print(fenwick_solution(nums), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


In [12]:
class SolutionFenwick:
    def maxFixedPoints(self, nums: list[int]) -> int:
        pairs = sorted((i - x, x) for i, x in enumerate(nums) if i >= x)
        if not pairs:
            return 0
        size = max(x for _, x in pairs) + 2
        t = [0] * (size + 1)
        ans = 0
        for _, x in pairs:
            i, best = x, 0
            while i > 0:
                if t[i] > best:
                    best = t[i]
                i -= i & -i
            best += 1
            i = x + 1
            while i <= size:
                if t[i] < best:
                    t[i] = best
                i += i & -i
            if best > ans:
                ans = best
        return ans

print([SolutionFenwick().maxFixedPoints(x) for x, _ in examples])

[2, 2, 3]


**Độ phức tạp Cách 4:** `O(n log n + n log V)` với `V ≤ 10^5`; bộ nhớ `O(n + V)`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [2, 0, 1, 1, 4, 2, 3]` (ví dụ tự dựng).

| i | x | d = i − x |
|---|---|---|
| 0 | 2 | −2 (loại) |
| 1 | 0 | 1 |
| 2 | 1 | 1 |
| 3 | 1 | 2 |
| 4 | 4 | 0 |
| 5 | 2 | 3 |
| 6 | 3 | 3 |

Sau khi sắp: `(0,4), (1,0), (1,1), (2,1), (3,2), (3,3)`. LIS tăng ngặt trên `x = [4, 0, 1, 1, 2, 3]` là `0,1,2,3` ⇒ đáp án `4`. Hàm trace in ra mảng `tails` sau mỗi bước.

In [13]:
def trace_lis(nums):
    pairs = sorted((i - x, x) for i, x in enumerate(nums) if i - x >= 0)
    print("cặp (d, x) sau khi sắp:", pairs)
    tails = []
    for step, (d, x) in enumerate(pairs, 1):
        pos = bisect_left(tails, x)
        action = "nối thêm" if pos == len(tails) else f"thay tails[{pos}]={tails[pos]}"
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
        print(f"bước {step}: (d={d}, x={x}) pos={pos} -> {action:<16} tails={tails}")
    print("đáp án =", len(tails))

trace_lis([2, 0, 1, 1, 4, 2, 3])

cặp (d, x) sau khi sắp: [(0, 4), (1, 0), (1, 1), (2, 1), (3, 2), (3, 3)]
bước 1: (d=0, x=4) pos=0 -> nối thêm         tails=[4]
bước 2: (d=1, x=0) pos=0 -> thay tails[0]=4  tails=[0]
bước 3: (d=1, x=1) pos=1 -> nối thêm         tails=[0, 1]
bước 4: (d=2, x=1) pos=1 -> thay tails[1]=1  tails=[0, 1]
bước 5: (d=3, x=2) pos=2 -> nối thêm         tails=[0, 1, 2]
bước 6: (d=3, x=3) pos=3 -> nối thêm         tails=[0, 1, 2, 3]
đáp án = 4


In [14]:
# Kiểm tra lại trace bằng vét cạn
print("vét cạn:", brute([2, 0, 1, 1, 4, 2, 3]))

vét cạn: 4


Trace dạng "dựng lại mảng": từ tập điểm bất động tìm được, in ra một cách xóa cụ thể để thấy chỉ số khớp.

In [15]:
def reconstruct(nums):
    """Trả về mảng sau khi xóa đạt số điểm bất động tối đa (dùng DP O(n^2) để truy vết)."""
    n = len(nums)
    d = [i - x for i, x in enumerate(nums)]
    dp, par = [0] * n, [-1] * n
    for i in range(n):
        if d[i] < 0:
            continue
        dp[i] = 1
        for j in range(i):
            if d[j] >= 0 and nums[j] < nums[i] and d[j] <= d[i] and dp[j] + 1 > dp[i]:
                dp[i], par[i] = dp[j] + 1, j
    if max(dp) == 0:
        return []
    end = max(range(n), key=lambda i: dp[i])
    chosen = []
    while end != -1:
        chosen.append(end); end = par[end]
    chosen.reverse()
    keep = set(chosen)
    # giữ đủ phần tử "đệm" giữa các điểm chọn
    prev_i, prev_x = -1, -1
    for i in chosen:
        need = nums[i] - prev_x - 1
        for j in range(prev_i + 1, i):
            if need == 0:
                break
            keep.add(j); need -= 1
        prev_i, prev_x = i, nums[i]
    result = [nums[i] for i in range(n) if i in keep]
    print("chọn điểm bất động tại chỉ số gốc:", chosen, "| mảng sau khi xóa:", result,
          "| số điểm bất động:", count_fixed(result))
    return result

reconstruct([2, 0, 1, 1, 4, 2, 3]);
reconstruct([1, 0, 1, 2]);

chọn điểm bất động tại chỉ số gốc: [1, 2, 5, 6] | mảng sau khi xóa: [0, 1, 2, 3] | số điểm bất động: 4
chọn điểm bất động tại chỉ số gốc: [1, 2, 3] | mảng sau khi xóa: [0, 1, 2] | số điểm bất động: 3


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maxFixedPoints(self, nums: list[int]) -> int:
        # Chỉ giữ các phần tử có thể thành điểm bất động: d = i - x >= 0
        pairs = sorted((i - x, x) for i, x in enumerate(nums) if i >= x)
        tails = []
        for _, x in pairs:
            pos = bisect_left(tails, x)
            if pos == len(tails):
                tails.append(x)
            else:
                tails[pos] = x
        return len(tails)

sol = Solution()
for nums, exp in examples:
    got = sol.maxFixedPoints(nums)
    print(nums, "->", got, "✔" if got == exp else "✘")

[0, 2, 1] -> 2 ✔
[3, 1, 2] -> 2 ✔
[1, 0, 1, 2] -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Chỉ cần chọn tập điểm bất động.** Mọi cách xóa cho ra một tập điểm bất động `S`. Ngược lại theo bổ đề ở mục 6, mọi tập `S` thỏa 3 điều kiện đều thực hiện được bằng cách giữ đúng số phần tử đệm. Nên đáp án = kích thước lớn nhất của tập thỏa 3 điều kiện.
2. **Cần:** nếu `i` là điểm bất động thì có đúng `x` phần tử giữ trước nó trong `i` phần tử gốc ⇒ `d ≥ 0`. Hai điểm bất động `i < j` có chỉ số mới `x_i < x_j`. Số phần tử bị xóa trước `j` = số bị xóa trước `i` + số bị xóa trong `[i, j)` ⇒ `d_j ≥ d_i`.
3. **Quên chỉ số:** `i = d + x`, nên `x` tăng ngặt và `d` không giảm kéo theo `i` tăng ngặt. Mọi dãy con của danh sách đã sắp theo `(d, x)` có `d` không giảm; chọn thêm `x` tăng ngặt thì thỏa cả 3 điều kiện. Ngược lại, mọi tập hợp lệ khi viết theo `i` tăng cũng có `(d, x)` tăng theo thứ tự từ điển, nên nó là một dãy con của danh sách đã sắp.
4. **LIS đúng:** thuật toán `tails` + `bisect_left` cho độ dài dãy con tăng ngặt dài nhất (bất biến: `tails[L]` là giá trị cuối nhỏ nhất của dãy tăng độ dài `L+1`, và `tails` tăng ngặt).

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `V = max(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP | `O(n²)` | `O(n)` |
| 3. Sắp xếp + LIS | `O(n log n)` | `O(n)` |
| 4. Fenwick | `O(n log n + n log V)` | `O(n + V)` |

Không có khác biệt average/worst đáng kể: mọi cách đều duyệt toàn bộ dữ liệu.

## 12. Bộ test

In [17]:
def run_all(nums):
    return (brute(nums) if len(nums) <= 12 else None, dp_quadratic(nums), lis_solution(nums),
            fenwick_solution(nums), SolutionQuadratic().maxFixedPoints(nums),
            SolutionFenwick().maxFixedPoints(nums), Solution().maxFixedPoints(nums))

# 12.1 ví dụ đề
for nums, exp in examples:
    r = run_all(nums)
    assert all(v == exp for v in r if v is not None), (nums, r)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [18]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
edge = [
    ([0], 1),              # một phần tử, đã bất động
    ([5], 0),              # không thể bất động
    ([0, 0, 0], 1),        # chỉ một số 0 được dùng
    ([0, 1, 2, 3], 4),     # đã hoàn hảo
    ([3, 2, 1, 0], 1),     # giảm dần: chỉ có 0 ở cuối (d=3)
    ([1, 1, 1, 1], 1),     # chỉ một số 1 dùng được
    ([0, 5, 5, 5, 5, 5, 1], 2),
]
for nums, exp in edge:
    r = run_all(nums)
    assert all(v == exp for v in r if v is not None), (nums, r, exp)
print("Ca biên: OK")

Ca biên: OK


In [19]:
# 12.3 phản ví dụ cho hai ý tưởng sai (tự tìm bằng seed cố định, kiểm chứng bằng vét cạn)
def greedy_count(nums):          # sai: đếm mọi x <= i
    return sum(1 for i, x in enumerate(nums) if x <= i)
def lis_ignore_d(nums):          # sai: chỉ đòi x tăng ngặt theo thứ tự i, quên điều kiện d
    return lis_strict([x for i, x in enumerate(nums) if x <= i])
rng = random.Random(123)
for name, f in (("đếm x<=i", greedy_count), ("bỏ qua d", lis_ignore_d)):
    for _ in range(5000):
        nums = [rng.randint(0, 5) for _ in range(rng.randint(1, 7))]
        if f(nums) != brute(nums):
            print(f"{name}: phản ví dụ {nums} -> sai {f(nums)}, đúng {Solution().maxFixedPoints(nums)} (vét cạn {brute(nums)})")
            break

đếm x<=i: phản ví dụ [0, 0, 3] -> sai 2, đúng 1 (vét cạn 1)
bỏ qua d: phản ví dụ [0, 4, 3, 0, 0, 2, 5] -> sai 3, đúng 2 (vét cạn 2)


In [20]:
# 12.4 đối chiếu ngẫu nhiên (seed cố định)
rng = random.Random(3920)
for _ in range(500):
    n = rng.randint(1, 11)
    nums = [rng.randint(0, n) for _ in range(n)]
    r = run_all(nums)
    assert len(set(r)) == 1, (nums, r)
print("500 test ngẫu nhiên nhỏ: OK")
for _ in range(100):
    n = rng.randint(50, 300)
    nums = [rng.randint(0, n) for _ in range(n)]
    r = run_all(nums)[1:]
    assert len(set(r)) == 1, nums
print("100 test cỡ vừa (so với DP O(n^2)): OK")

500 test ngẫu nhiên nhỏ: OK


100 test cỡ vừa (so với DP O(n^2)): OK


In [21]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(1)
big = [rng.randint(0, 10**5) for _ in range(10**5)]
t0 = time.time(); a = Solution().maxFixedPoints(big); t1 = time.time()
b = SolutionFenwick().maxFixedPoints(big)
print("đáp án:", a, "| khớp Fenwick:", a == b, "| nhanh (<2s):", t1 - t0 < 2)
big2 = list(range(10**5))
print("mảng 0..n-1:", Solution().maxFixedPoints(big2))

đáp án: 312 | khớp Fenwick: True | nhanh (<2s): True


mảng 0..n-1: 100000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp `x` giảm dần khi `d` bằng nhau (theo thói quen LIS 2D "phong bì")

Ở bài *Russian Doll Envelopes* (LC 354), khi khóa thứ nhất bằng nhau ta sắp khóa thứ hai **giảm dần** để không chọn hai phong bì cùng chiều rộng. Nhưng ở đây `d` chỉ cần **không giảm** (được phép bằng nhau), nên phải sắp `x` **tăng dần**.

In [22]:
def wrong_desc(nums):
    pairs = sorted(((i - x, x) for i, x in enumerate(nums) if i >= x), key=lambda p: (p[0], -p[1]))
    return lis_strict([x for _, x in pairs])

nums = [1, 0, 1, 2]
print("sai:", wrong_desc(nums), "| đúng:", Solution().maxFixedPoints(nums))

sai: 1 | đúng: 3


**Vì sao sai:** với `d = 1` cho cả ba phần tử `x = 0,1,2`, sắp giảm thành `2,1,0` nên LIS chỉ lấy được 1. **Sửa:** sắp `(d, x)` tăng dần theo cả hai khóa.

### Lỗi 2 — Dùng `bisect_right` (dãy không giảm) thay vì `bisect_left`

Hai điểm bất động không thể có cùng giá trị `x` (cùng chỉ số mới). Dùng `bisect_right` sẽ cho phép `x` bằng nhau.

In [23]:
from bisect import bisect_right
def wrong_right(nums):
    pairs = sorted((i - x, x) for i, x in enumerate(nums) if i >= x)
    tails = []
    for _, x in pairs:
        pos = bisect_right(tails, x)
        if pos == len(tails): tails.append(x)
        else: tails[pos] = x
    return len(tails)

nums = [0, 0, 0]
print("sai:", wrong_right(nums), "| đúng:", Solution().maxFixedPoints(nums))

sai: 3 | đúng: 1


### Lỗi 3 — Quên lọc `d < 0`

Phần tử có `x > i` không bao giờ thành điểm bất động (trước nó không đủ phần tử). Nếu quên lọc, LIS có thể chọn nó.

In [24]:
def wrong_no_filter(nums):
    pairs = sorted((i - x, x) for i, x in enumerate(nums))
    return lis_strict([x for _, x in pairs])

nums = [1, 3, 5]
print("sai:", wrong_no_filter(nums), "| đúng:", Solution().maxFixedPoints(nums))

sai: 1 | đúng: 0


**Vì sao sai:** với `[1, 3, 5]` không phần tử nào có `x ≤ i` nên đáp án là `0`, nhưng khi quên lọc, LIS vẫn chọn được 1 phần tử. Ô dưới đây tìm thêm một phản ví dụ ngẫu nhiên. **Sửa:** luôn lọc `i >= x` trước khi chạy LIS.

In [25]:
rng = random.Random(7)
diff = None
for _ in range(2000):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(1, 7))]
    if wrong_no_filter(nums) != Solution().maxFixedPoints(nums):
        diff = nums; break
print("ví dụ làm lộ lỗi 3:", diff, "| sai:", wrong_no_filter(diff), "| đúng:", Solution().maxFixedPoints(diff))

ví dụ làm lộ lỗi 3: [1, 3, 5] | sai: 1 | đúng: 0


### Lỗi 4 — Chỉ đếm điểm bất động "sẵn có" rồi xóa tham lam từ trái

Tham lam "gặp `x < chỉ_số_hiện_tại` thì xóa" có thể phá hỏng các điểm bất động phía sau. Bài này cần tối ưu toàn cục (LIS), không có tham lam đơn giản.

In [26]:
def greedy_delete(nums):
    res = []
    for x in nums:
        if x >= len(res):          # giữ nếu còn hy vọng thành bất động
            res.append(x)
    return count_fixed(res)

for nums in ([3, 1, 2], [0, 2, 1], [1, 0, 1, 2]):
    print(nums, "tham lam:", greedy_delete(nums), "| đúng:", Solution().maxFixedPoints(nums))

[3, 1, 2] tham lam: 2 | đúng: 2
[0, 2, 1] tham lam: 1 | đúng: 2
[1, 0, 1, 2] tham lam: 2 | đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "**xóa phần tử, chỉ số dồn lại**" ⇒ nghĩ theo **dãy con được giữ**, và tìm đại lượng "số phần tử bị xóa trước vị trí này" (`i - chỉ_số_mới`).
- Hai điều kiện đơn điệu đồng thời (một **ngặt**, một **không ngặt**) ⇒ **LIS hai chiều**: sắp theo chiều thứ nhất, LIS theo chiều thứ hai. Cẩn thận chiều sắp phụ khi bằng nhau:
  - chiều 1 **cho phép bằng** ⇒ sắp chiều 2 **tăng** (để nối được nhau);
  - chiều 1 **phải ngặt** ⇒ sắp chiều 2 **giảm** (để không nối được nhau).
- Kiểm tra nhanh: nếu tìm được `i = d + x` (biến thứ ba là tổng/hiệu của hai biến kia) thì điều kiện thứ ba thường **tự động** thỏa.

## 15. Biến thể của bài

1. **Điểm bất động 1-indexed** (`nums[i] == i + 1`): chỉ đổi `d = i + 1 - x`.
2. **Chỉ được xóa tối đa `m` phần tử**: thêm điều kiện `d ≤ m` cho mọi điểm được chọn (vì `d` không giảm, chỉ cần điểm cuối `d ≤ m`).
3. **Đếm số cách xóa** đạt tối đa: LIS đếm số cách (Fenwick lưu cặp `(độ dài, số cách)`).
4. **Trả về mảng sau khi xóa**: truy vết LIS rồi chèn phần tử đệm như hàm `reconstruct`.
5. Họ bài liên quan: *Longest Increasing Subsequence* (LC 300), *Russian Doll Envelopes* (LC 354), *Maximum Height by Stacking Cuboids* (LC 1691).

In [27]:
# Biến thể 2: xóa tối đa m phần tử
def with_budget(nums, m):
    pairs = sorted((i - x, x) for i, x in enumerate(nums) if 0 <= i - x <= m)
    return lis_strict([x for _, x in pairs])

def brute_budget(nums, m):
    n = len(nums); best = 0
    for mask in range(1 << n):
        if n - bin(mask).count('1') <= m:
            best = max(best, count_fixed([nums[i] for i in range(n) if mask >> i & 1]))
    return best

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(1, 9))]
    m = rng.randint(0, 5)
    assert with_budget(nums, m) == brute_budget(nums, m)
print("Biến thể 2 khớp vét cạn.")

Biến thể 2 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi tập giữ lại | `O(2^n n)` | `O(n)` | Kiểm thử, `n ≤ 15` |
| 2. DP `O(n²)` | `dp[i]` kết thúc ở `i` | `O(n²)` | `O(n)` | Hiểu bài, `n ≤ 3000` |
| 3. Sắp xếp + LIS | LIS 2D với `bisect` | `O(n log n)` | `O(n)` | **Lời giải chuẩn**, code ngắn |
| 4. Fenwick max | DP theo thứ tự `d` + BIT | `O(n log V)` | `O(n+V)` | Khi cần mở rộng (đếm số cách, trọng số) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxFixedPoints([0, 0, 1, 1, 2, 2])` = ?

**Bài 2 (tính tay).** Với `nums = [2, 0, 3, 1, 2]`, lập bảng `(i, x, d)` và tính đáp án bằng tay.

**Bài 3 (điền TODO).** Hoàn thiện hàm `pairs_of(nums)` trả về danh sách cặp `(d, x)` đã sắp, đã lọc.

**Bài 4 (tự viết lại).** Viết lại lời giải nhưng LIS dùng Fenwick (không xem Cách 4).

**Bài 5 (mini challenge).** Mỗi phần tử có thêm trọng số `w[i] > 0`. Tìm **tổng trọng số lớn nhất** của các điểm bất động đạt được. (Gợi ý: Fenwick max lưu tổng trọng số.)

In [28]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def pairs_of(nums):
    # TODO: tạo (i - x, x) cho các i có i - x >= 0, rồi sắp xếp
    pass

In [29]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_weight_fixed(nums, w):
    # TODO: sắp (d, x) tăng dần, Fenwick max theo x lưu tổng trọng số
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Lập `d = i - x`: `0,1,1,2,2,3`. (2) Sắp `(d,x)`: `(0,0),(1,0),(1,1),(2,1),(2,2),(3,2)`. (3) LIS ngặt trên `0,0,1,1,2,2` → 3.

**Bài 2.** (1) `d = −2, 1, −1, 2, 2`. (2) Cặp dùng được: `(1,0),(2,1),(2,2)`. (3) LIS → 3.

**Bài 3.** (1) Dùng `enumerate`. (2) Điều kiện `i >= x`. (3) `sorted(...)`.

**Bài 4.** (1) Chỉ số Fenwick = `x + 1`. (2) `query(x)` lấy max của giá trị `< x`. (3) `update(x+1, best)`.

**Bài 5.** (1) Thay `+1` bằng `+w[i]`. (2) Cần giữ `w` cùng cặp: `(d, x, w)`. (3) Đáp án là max mọi `best`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, 2
assert Solution().maxFixedPoints([0, 0, 1, 1, 2, 2]) == 3 == brute([0, 0, 1, 1, 2, 2])
assert Solution().maxFixedPoints([2, 0, 3, 1, 2]) == 3 == brute([2, 0, 3, 1, 2])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [31]:
# Bài 3
def pairs_of_sol(nums):
    return sorted((i - x, x) for i, x in enumerate(nums) if i - x >= 0)
assert pairs_of_sol([2, 0, 3, 1, 2]) == [(1, 0), (2, 1), (2, 2)]
print("Bài 3 OK:", pairs_of_sol([2, 0, 3, 1, 2]))

Bài 3 OK: [(1, 0), (2, 1), (2, 2)]


In [32]:
# Bài 4
def sol_bit(nums):
    pairs = pairs_of_sol(nums)
    if not pairs:
        return 0
    fw = FenwickMax(max(x for _, x in pairs) + 2)
    ans = 0
    for _, x in pairs:
        b = fw.query(x) + 1
        fw.update(x + 1, b)
        ans = max(ans, b)
    return ans

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(0, 8) for _ in range(rng.randint(1, 10))]
    assert sol_bit(nums) == Solution().maxFixedPoints(nums)
print("Bài 4 OK")

Bài 4 OK


In [33]:
# Bài 5
def max_weight_fixed_sol(nums, w):
    trip = sorted((i - x, x, w[i]) for i, x in enumerate(nums) if i >= x)
    if not trip:
        return 0
    fw = FenwickMax(max(x for _, x, _ in trip) + 2)
    ans = 0
    for _, x, wt in trip:
        b = fw.query(x) + wt
        fw.update(x + 1, b)
        ans = max(ans, b)
    return ans

def brute_weight(nums, w):
    n = len(nums); best = 0
    for mask in range(1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        best = max(best, sum(w[i] for pos, i in enumerate(idx) if nums[i] == pos))
    return best

rng = random.Random(55)
for _ in range(300):
    n = rng.randint(1, 9)
    nums = [rng.randint(0, 6) for _ in range(n)]
    w = [rng.randint(1, 9) for _ in range(n)]
    assert max_weight_fixed_sol(nums, w) == brute_weight(nums, w)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# LIS hai chiều: d không giảm, x tăng ngặt
pairs = sorted((i - x, x) for i, x in enumerate(nums) if i >= x)
tails = []
for _, x in pairs:
    p = bisect_left(tails, x)          # ngặt -> bisect_left
    if p == len(tails): tails.append(x)
    else: tails[p] = x
return len(tails)
```

**Checklist:**

- [ ] Đã lọc các phần tử có `x > i` (không thể bất động)?
- [ ] Sắp `(d, x)` **tăng** theo cả hai khóa (vì `d` được phép bằng nhau)?
- [ ] LIS **tăng ngặt** theo `x` (`bisect_left`)?
- [ ] Trường hợp không có phần tử nào dùng được → trả `0`?
- [ ] Độ phức tạp `O(n log n)` cho `n = 10^5`?